# Classification finale du corpus TASS avec un LLM

Ce notebook applique au corpus complet une classification binaire destinée à distinguer les articles relevant du thème étudié de ceux qui doivent être exclus de l’échantillon final.

La catégorie `hors_echantillon = false` correspond aux articles à conserver : ils portent sur les déplacements, l’arrivée, l’accueil ou la prise en charge en Russie de personnes venues d’Ukraine, du Donbass ou de territoires sous contrôle russe. La catégorie `hors_echantillon = true` correspond aux articles à exclure : ils ne traitent pas directement de cette migration vers la Russie, ou portent sur d’autres formes d’aide, de déplacement ou de contexte politique.

Ce notebook intervient après la phase de test des modèles sur un jeu annoté. Il reprend le prompt retenu et l’applique à l’ensemble du corpus afin de produire deux fichiers séparés : un corpus `dans_echantillon`, utilisé pour l’analyse, et un corpus `hors_echantillon`, conservé comme sortie de contrôle.


## 1. Préparation du corpus complet

Cette première partie charge le corpus complet au format JSON. Chaque entrée correspond à un article TASS déjà préparé pour l’annotation par LLM, avec au minimum un identifiant et le texte de l’article.


In [2]:
import json 

In [11]:
with open("/Users/quentinnippert/Documents/mm_files/TASS_analyse/data_llm_questions_full.json", "r", encoding="utf-8") as f:
    full_corpus = json.load(f)

In [12]:
len(full_corpus)

1870

In [7]:
full_corpus 

[{'id': 1,
  'text': 'МОСКВА, 19 февраля. /ТАСС/. Волонтеры Общероссийского народного фронта (ОНФ), Всероссийского студенческого корпуса спасателей, Российского Красного Креста и других добровольческих объединений формируют штаб "Мы вместе" для оказания поддержки прибывающим на территорию России из Донецкой и Луганской народных республик (ДНР и ЛНР). Об этом в субботу сообщила пресс-служба ОНФ.\n"Сейчас в экстренном порядке разворачивается штаб помощи для того, чтобы поддержать прибывших жителей ДНР и ЛНР, обеспечить их предметами первой необходимости, продуктами, оказать психологическую помощь. Конечно, первыми их будут встречать спасатели и местные власти, но мы должны быстро скоординировать общественную поддержку в Ростове-на-Дону, а если понадобится, то и по всей стране", - сказал руководитель исполкома Народного фронта Михаил Кузнецов.\nПресс-служба уточнила, что круглосуточный оперативный волонтерский штаб будет координировать работу по сбору и распределению гуманитарной помощи, 

## 2. Conversion du corpus en JSONL

Le corpus est ensuite converti en format JSONL, c’est-à-dire un objet JSON par ligne. Ce format est plus adapté à un traitement incrémental : il permet de lire et d’annoter les articles un par un, sans devoir recharger ou réécrire tout le fichier à chaque étape.


In [13]:
output_path = "/Users/quentinnippert/Documents/mm_files/TASS_analyse/data_llm_questions_full.jsonl"


with open(output_path, "w", encoding="utf-8") as f:
    for item in full_corpus:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Saved:", output_path)
print("Nombre d'articles:", len(full_corpus))

Saved: /Users/quentinnippert/Documents/mm_files/TASS_analyse/data_llm_questions_full.jsonl
Nombre d'articles: 1870


## 3. Lancement de la classification sur le corpus complet

Cette section met en place la procédure d’annotation automatique appliquée à l’ensemble du corpus. Elle définit les bibliothèques nécessaires, le prompt d’annotation, les chemins des fichiers d’entrée et de sortie, puis la boucle d’appel au modèle.

La logique générale est la suivante : pour chaque article non encore traité, le modèle reçoit le texte et doit répondre uniquement par un JSON contenant l’identifiant de l’article et la valeur du label `hors_echantillon`.


In [20]:
import json
import re
import time
import requests
from pathlib import Path

### 3.1. Définition du prompt d’annotation

Le prompt explicite les critères de décision entre les deux classes. La valeur `hors_echantillon = false` signifie que l’article doit rester dans l’échantillon d’analyse, car il traite de la migration de personnes venues d’Ukraine, du Donbass ou de territoires sous contrôle russe vers la Russie, la Crimée ou l’Ossétie du Sud.

La valeur `hors_echantillon = true` signifie que l’article doit être exclu : il traite d’un autre sujet, d’une aide humanitaire sans déplacement vers la Russie, de déplacements internes, de pays tiers, ou d’enjeux politiques et administratifs non directement liés à l’accueil en Russie.

Le format de réponse est volontairement contraint afin de faciliter le parsing automatique des sorties du modèle.


In [21]:
SYSTEM_PROMPT = """
Tu es un spécialiste de l’annotation de discours médiatiques.

Ta tâche est uniquement de déterminer si un article TASS en russe doit recevoir le label suivant :

"Article hors échantillon - ne parle pas de la migration de l'Ukraine vers la Russie"

========================
HORS_ECHANTILLON = FALSE 
========================

Choisis hors_echantillon = false si l’article traite principalement de la migration des gens provenants de l'Ukraine, Donetsk, Louhansk vers la Russie, notamment en ce qui concerne :

- le déplacement vers la Russie ;
- l’arrivée en Russie ;
- les évacuations vers la Russie ;
- l'ouverture des couloirs humanitaires par la Russie en Ukraine pour évacuer des civils vers la Russie ;
- l’accueil en Russie ;
- l’installation en Russie ;
- l’hébergement en Russie ;
- la prise en charge en Russie ;
- les conditions de vie de ces gens d'Ukraine en Russie ;
- les PVR, centres d’accueil, points d’hébergement ou structures de placement pour ces gens en Russie ;
- l’aide administrative, médicale, scolaire, sociale ou financière pour ces gens en Russie ;
- les lois, décisions, des dispositifs, les mesures sociales, juridiques, financières ou administratives destinées aux évacués, réfugiés, déplacés ou migrants venus d’Ukraine / Donbass, même si l’article est présenté comme une réunion politique ou institutionnelle ;
- l’envoi d’aide humanitaire vers la Russie, Rostov, Belgorod, la Crimée ou d’autres régions russes pour des réfugiés, déplacés ou évacués ;
- les enfants venant d’Ukraine, du Donbass, de la DNR, de la LNR, de Kherson, de Zaporijjia ou de régions sous contrôle russe accueillis en Russie, en Crimée ou en Ossétie du Sud.

IMPORTANT :
- Une évacuation vers la Russie = toujours false.
- Une évacuation vers la Crimée ou l’Ossétie du Sud = false.
- L’accueil en Russie d’enfants, familles ou civils venus d’Ukraine / Donbass = false.
- Tout article sur des PVR en Russie pour ces personnes = false.
- Tout article sur des enfants venus d’Ukraine / Donbass présents en Russie, en Crimée ou en Ossétie du Sud = false.
- Les couloirs humanitaires ouverts par la Russie en Ukraine pour évacuer des civils = false, même si l’évacuation commence en Ukraine.

Exemples false :
- « Вторая партия гуманитарного груза для эвакуированных из Донецкой и Луганской народных республик отправлена в Ростов-на-Дону. »
- « Крым готов принять порядка 5 тыс. детей из Херсонской области. »
- « Российские детские лагеря отдыха приняли детей из Донбасса и с освобожденных территорий Украины. »
- « Россия открыла гуманитарный коридор для эвакуации мирных жителей. »
- « Беженцам, прибывающим в российские регионы, выплачивают по 10 тыс. рублей. »

========================
HORS_ECHANTILLON = TRUE
========================

Réponds hors_echantillon = true si l’article porte principalement sur :

- l’accueil uniquement en Europe, en Pologne, en Allemagne ou dans un autre pays tiers, sans Russie / Crimée / Ossétie du Sud ;
- une aide humanitaire envoyée aux habitants du Donbass, aux habitants de la DNR/LNR, de Kherson, de Zaporijjia ou d’autres territoires, sans mention d’évacués, réfugiés ou déplacés vers la Russie ;
- une aide humanitaire formulée comme aide aux “habitants du Donbass” / “habitants de la DNR et de la LNR” = true ;
- des déplacés internes en Ukraine ou dans les territoires occupés, sans arrivée en Russie ;
- des déplacés internes de Russie vers d’autres régions russes, sans lien avec l’Ukraine / Donbass ;
- des évacuations à l’intérieur de l’Ukraine ou vers des territoires non russes ;
- une évacuation de l’Ukraine vers une autre partie de l’Ukraine ;
- la mobilisation, les combats, les opérations militaires, les bombardements ou la situation politique, sans accueil ni prise en charge en Russie ;
- l’aide à des militaires ou à des participants de l’opération militaire, et non à des réfugiés / déplacés / évacués ;
- des catastrophes naturelles ou événements internes en Russie sans lien avec la migration depuis l’Ukraine / Donbass vers la Russie ;
- des dispositifs politiques, administratifs ou civiques liés aux passeports, à la citoyenneté, aux référendums, aux élections, aux bureaux de vote ou au statut des territoires. 

IMPORTANT :
- “Aide aux habitants du Donbass / DNR / LNR” = true.
- Évacuation Ukraine → Ukraine = true.
- Si l’article parle uniquement de pays voisins ou de l’Europe, répondre true.
- Si l’article parle de plusieurs pays et inclut la Russie parmi les destinations, répondre true

Exemples true :
- « Около 80 тонн гуманитарной помощи для жителей Донецкой и Луганской народных республик отправили из Дагестана. »
- « Фура с продуктами, собранными для жителей Донбасса, отправилась в Ростовскую область » si le texte présente l’aide comme destinée aux habitants du Donbass et non aux évacués en Russie.
- « Восемь зарубежных избирательных участков организуют в Якутии для проведения референдумов... »
- « Крым окажет содействие в оформлении российских паспортов для жителей освобожденных территорий Украины. »
- « Две автобусные колонны с эвакуированными жителями города Сумы приехали в Полтавскую область. »

========================
FORMAT
========================

Réponds uniquement en JSON valide :

{"id": ..., "hors_echantillon": true}

ou

{"id": ..., "hors_echantillon": false}
"""

USER_PROMPT = """
Lis l’article TASS ci-dessous et détermine uniquement s’il doit être classé comme hors échantillon.

Retourne uniquement un JSON valide.
"""

### 3.2. Chemins des fichiers et choix du modèle

Cette cellule définit le fichier JSONL d’entrée, le fichier de résultats et le fichier d’erreurs. Les résultats sont enregistrés progressivement dans un fichier JSONL afin que le traitement puisse être repris après une interruption.

Le modèle utilisé est celui retenu pour cette phase de classification finale après les tests réalisés sur l’échantillon de validation.


In [7]:
INPUT_JSONL = "/Users/quentinnippert/Documents/mm_files/TASS_analyse/data_llm_questions_full.jsonl"
OUTPUT_JSONL = "/Users/quentinnippert/Documents/mm_files/TASS_analyse/1LLM_HORS-DANS_echantillon/hors_full_results.jsonl"
ERROR_JSONL = "/Users/quentinnippert/Documents/mm_files/TASS_analyse/1LLM_HORS-DANS_echantillon/hors_full_errors.jsonl"

MODEL = "openai/gpt-5.4-mini"

### 3.3. Connexion à l’API

Cette cellule configure l’accès à l’API utilisée pour interroger le modèle. La clé API doit être renseignée localement avant l’exécution du notebook. Dans la suite du code, les requêtes sont envoyées article par article au modèle configuré.


In [ ]:
API_URL = "https://openrouter.ai/api/v1/chat/completions"
API_KEY = ""  # <-- replace with your actual API key
client = OpenAI(api_key=API_KEY, base_url=API_URL) 

### 3.4. Boucle d’annotation et sauvegarde incrémentale

Cette cellule constitue le cœur du notebook. Elle parcourt le corpus ligne par ligne, ignore les articles déjà traités, envoie chaque texte au modèle, extrait le JSON retourné, puis sauvegarde immédiatement le résultat.

Le fonctionnement par `done_ids` permet de reprendre l’annotation sans recommencer depuis le début. Les erreurs éventuelles sont écrites dans un fichier séparé, ce qui permet de vérifier ensuite les articles non traités ou les réponses mal formées.


In [25]:
results = []

def extract_json(text):
    if text is None:
        raise ValueError("Response content is None")

    text = text.strip()

    if text.startswith("```"):
        text = re.sub(r"^```json", "", text).strip()
        text = re.sub(r"^```", "", text).strip()
        text = re.sub(r"```$", "", text).strip()

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        raise ValueError(f"No JSON found in response: {text}")

    return json.loads(match.group(0))


def append_jsonl(path, obj):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")
        f.flush()


def load_done_ids(output_path):
    done = set()

    if not Path(output_path).exists():
        return done

    with open(output_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try:
                    item = json.loads(line)
                    done.add(item["id"])
                except Exception:
                    pass

    return done


done_ids = load_done_ids(OUTPUT_JSONL)
print("Already processed:", len(done_ids))


with open(INPUT_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        article = json.loads(line)
        article_id = article["id"]

        if article_id in done_ids:
            continue

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": (
                    f"{USER_PROMPT}\n\n"
                    f"ID: {article_id}\n\n"
                    f"{article['text']}"
                )
            }
        ]

        try:
            response = requests.post(
                "https://openrouter.ai/api/v1/chat/completions",
                headers={
                    "Authorization": f"Bearer {API_KEY}",
                    "Content-Type": "application/json"
                },
                json={
                    "model": MODEL,
                    "messages": messages,
                    "temperature": 0
                },
                timeout=120
            )

            response.raise_for_status()
            data = response.json()

            message = data["choices"][0]["message"]
            content = message.get("content")

            parsed = extract_json(content)

            result = {
                "id": article_id,
                "hors_echantillon": bool(parsed["hors_echantillon"])
            }

            results.append(result)
            append_jsonl(OUTPUT_JSONL, result)
            done_ids.add(article_id)

            print(f"Processed: {article_id} -> hors_echantillon={result['hors_echantillon']}")

            time.sleep(0.2)

        except Exception as e:
            error_item = {
                "id": article_id,
                "error": str(e),
                "response_text": None
            }

            try:
                error_item["response_text"] = response.text
            except Exception:
                pass

            append_jsonl(ERROR_JSONL, error_item)

            print(f"ERROR article {article_id}: {e}")
            time.sleep(2)

print("Done.")
print("New results in memory:", len(results))
print("Total saved:", len(load_done_ids(OUTPUT_JSONL)))

Already processed: 0
Processed: 1 -> hors_echantillon=False
Processed: 2 -> hors_echantillon=False
Processed: 3 -> hors_echantillon=False
Processed: 4 -> hors_echantillon=False
Processed: 5 -> hors_echantillon=False
Processed: 6 -> hors_echantillon=False
Processed: 7 -> hors_echantillon=False
Processed: 8 -> hors_echantillon=False
Processed: 9 -> hors_echantillon=False
Processed: 10 -> hors_echantillon=False
Processed: 11 -> hors_echantillon=False
Processed: 12 -> hors_echantillon=False
Processed: 13 -> hors_echantillon=False
Processed: 14 -> hors_echantillon=False
Processed: 15 -> hors_echantillon=False
Processed: 16 -> hors_echantillon=False
Processed: 17 -> hors_echantillon=False
Processed: 18 -> hors_echantillon=False
Processed: 19 -> hors_echantillon=False
Processed: 20 -> hors_echantillon=False
Processed: 21 -> hors_echantillon=False
Processed: 22 -> hors_echantillon=False
Processed: 23 -> hors_echantillon=False
Processed: 24 -> hors_echantillon=False
Processed: 25 -> hors_echant

## 4. Vérification des résultats produits

Après l’annotation, cette section recharge le fichier de résultats afin de vérifier que les sorties ont bien été enregistrées. Elle permet de contrôler le nombre total d’articles classés et d’inspecter les premières lignes du fichier de sortie.


In [26]:
with open(OUTPUT_JSONL, "r", encoding="utf-8") as f:
    results = [json.loads(line) for line in f if line.strip()]

In [27]:
results

[{'id': 1, 'hors_echantillon': False},
 {'id': 2, 'hors_echantillon': False},
 {'id': 3, 'hors_echantillon': False},
 {'id': 4, 'hors_echantillon': False},
 {'id': 5, 'hors_echantillon': False},
 {'id': 6, 'hors_echantillon': False},
 {'id': 7, 'hors_echantillon': False},
 {'id': 8, 'hors_echantillon': False},
 {'id': 9, 'hors_echantillon': False},
 {'id': 10, 'hors_echantillon': False},
 {'id': 11, 'hors_echantillon': False},
 {'id': 12, 'hors_echantillon': False},
 {'id': 13, 'hors_echantillon': False},
 {'id': 14, 'hors_echantillon': False},
 {'id': 15, 'hors_echantillon': False},
 {'id': 16, 'hors_echantillon': False},
 {'id': 17, 'hors_echantillon': False},
 {'id': 18, 'hors_echantillon': False},
 {'id': 19, 'hors_echantillon': False},
 {'id': 20, 'hors_echantillon': False},
 {'id': 21, 'hors_echantillon': False},
 {'id': 22, 'hors_echantillon': False},
 {'id': 23, 'hors_echantillon': False},
 {'id': 24, 'hors_echantillon': False},
 {'id': 25, 'hors_echantillon': False},
 {'id': 2

In [28]:
len(results)

1870

## 5. Séparation du corpus final en deux sous-corpus

À partir des résultats de classification, le notebook reconstruit deux ensembles d’articles : les articles exclus de l’échantillon (`hors_echantillon = true`) et les articles conservés pour l’analyse (`hors_echantillon = false`).

Cette séparation permet de garder une trace des exclusions tout en produisant le corpus final qui sera utilisé dans les étapes suivantes de l’analyse.


### 5.1. Extraction des articles hors échantillon

Cette étape identifie les articles classés comme `hors_echantillon = true`, puis récupère dans le corpus complet les textes correspondant à ces identifiants. Le fichier obtenu sert de contrôle méthodologique : il permet de vérifier quels articles ont été écartés par le modèle.


In [8]:
#load jsonl result 

with open(OUTPUT_JSONL, "r", encoding="utf-8") as f:
    results = [json.loads(line) for line in f if line.strip()]

results 

[{'id': 1, 'hors_echantillon': False},
 {'id': 2, 'hors_echantillon': False},
 {'id': 3, 'hors_echantillon': False},
 {'id': 4, 'hors_echantillon': False},
 {'id': 5, 'hors_echantillon': False},
 {'id': 6, 'hors_echantillon': False},
 {'id': 7, 'hors_echantillon': False},
 {'id': 8, 'hors_echantillon': False},
 {'id': 9, 'hors_echantillon': False},
 {'id': 10, 'hors_echantillon': False},
 {'id': 11, 'hors_echantillon': False},
 {'id': 12, 'hors_echantillon': False},
 {'id': 13, 'hors_echantillon': False},
 {'id': 14, 'hors_echantillon': False},
 {'id': 15, 'hors_echantillon': False},
 {'id': 16, 'hors_echantillon': False},
 {'id': 17, 'hors_echantillon': False},
 {'id': 18, 'hors_echantillon': False},
 {'id': 19, 'hors_echantillon': False},
 {'id': 20, 'hors_echantillon': False},
 {'id': 21, 'hors_echantillon': False},
 {'id': 22, 'hors_echantillon': False},
 {'id': 23, 'hors_echantillon': False},
 {'id': 24, 'hors_echantillon': False},
 {'id': 25, 'hors_echantillon': False},
 {'id': 2

In [9]:
#number of hors_echantillon = true et false

true_count = sum(1 for r in results if r.get("hors_echantillon") is True)
true_count

613

In [13]:
true_ids = {
    item["id"]
    for item in results
    if item.get("hors_echantillon") is True
}

full_corpus_hors_echantillon = [
    item
    for item in full_corpus
    if item["id"] in true_ids
]

print("Hors échantillon:", len(full_corpus_hors_echantillon))
print(full_corpus_hors_echantillon[:2])

Hors échantillon: 613
[{'id': 229, 'text': 'ЭЛИСТА, 22 февраля. /ТАСС/. Гуманитарный груз для эвакуированных из Донецкой и Луганской народных республик (ДНР и ЛНР) отправлен во вторник из республиканского центра Калмыкии - Элисты - в Ростов-на-Дону, сообщил в своем аккаунте в Instagram глава республики Бату Хасиков.\n"Калмыкия направила две грузовые машины с гуманитарным грузом в Ростовскую область. Мы собрали 20 тонн гуманитарной помощи для беженцев из ДНР и ЛНР. Это продовольственная продукция калмыцких производителей и товары первой необходимости", - написал он.\nВ пресс-службе главы региона уточнили, что гуманитарный груз, в частности, включает калмыцкий рис, тушенку и консервы собственного производства, бытовую технику.\nХасиков поблагодарил предпринимателей, власти всех муниципалитетов республики, депутата Государственной думы от Калмыкии Бадму Башанкаева, жителей региона, а также команду волонтеров, которые стали участниками благотворительной миссии.\nСитуация на линии соприкосн

In [14]:
with open ("/Users/quentinnippert/Documents/mm_files/TASS_analyse/1LLM_HORS-DANS_echantillon/llm_full_corpus_hors_echantillon.json", "w", encoding="utf-8") as f:
    json.dump(full_corpus_hors_echantillon, f, ensure_ascii=False, indent=2)

### 5.2. Extraction des articles dans l’échantillon

Cette étape sélectionne les articles classés comme `hors_echantillon = false`. Il s’agit du corpus pertinent pour l’analyse, c’est-à-dire les articles TASS portant sur la migration, l’évacuation, l’accueil ou la prise en charge en Russie des personnes venues d’Ukraine ou du Donbass.

Le résultat est sauvegardé dans un fichier JSON distinct, qui constitue la base principale pour les analyses ultérieures.


In [15]:
false_ids = {
    item["id"]
    for item in results
    if item.get("hors_echantillon") is False
}

full_corpus_dans_echantillon = [
    item
    for item in full_corpus
    if item["id"] in false_ids
]

print("Dans échantillon:", len(full_corpus_dans_echantillon))
print(full_corpus_dans_echantillon[:2])

Dans échantillon: 1257
[{'id': 1, 'text': 'МОСКВА, 19 февраля. /ТАСС/. Волонтеры Общероссийского народного фронта (ОНФ), Всероссийского студенческого корпуса спасателей, Российского Красного Креста и других добровольческих объединений формируют штаб "Мы вместе" для оказания поддержки прибывающим на территорию России из Донецкой и Луганской народных республик (ДНР и ЛНР). Об этом в субботу сообщила пресс-служба ОНФ.\n"Сейчас в экстренном порядке разворачивается штаб помощи для того, чтобы поддержать прибывших жителей ДНР и ЛНР, обеспечить их предметами первой необходимости, продуктами, оказать психологическую помощь. Конечно, первыми их будут встречать спасатели и местные власти, но мы должны быстро скоординировать общественную поддержку в Ростове-на-Дону, а если понадобится, то и по всей стране", - сказал руководитель исполкома Народного фронта Михаил Кузнецов.\nПресс-служба уточнила, что круглосуточный оперативный волонтерский штаб будет координировать работу по сбору и распределению 

In [17]:
with open ("/Users/quentinnippert/Documents/mm_files/TASS_analyse/1LLM_HORS-DANS_echantillon/llm_full_corpus_dans_echantillon.json", "w", encoding="utf-8") as f:
    json.dump(full_corpus_dans_echantillon, f, ensure_ascii=False, indent=2)